# Part 1 — The Augmented LLM: a real loop with typed tools

*Agents from First Principles, Part 1.*

The RAG series ended at Part 19 by building a reason/act/observe loop with four tools. It worked, but it never named the primitive underneath it, and its tools were plain functions called through text actions with no contract. Two failures follow from that: the model can name a tool that does not exist, or pass a malformed argument, and a text-parsed action layer accepts both silently and only explodes when the function is finally called.

This part names the primitive and gives the tools a contract.

**The Augmented LLM** (the primitive, after Anthropic's "Building Effective Agents"): a single model call, ringed by tools (and, later, memory and retrieval), wrapped in the smallest loop with an explicit stop condition. Everything in this whole series is this primitive with one more ring added per part.

**The Ladder** -- "do you even need an agent?" Three rungs, same question, rising power and rising cost. Reach for the lowest rung that works:
1. **One augmented call**: the model may call tools, but only one round, then it must answer. Cheapest. Cannot chain a step whose input is the previous step's output.
2. **Fixed workflow**: a hardcoded sequence of calls. Predictable and cheap, but the route is wired at author time.
3. **Full agent**: the reason/act/observe loop. The model picks the next tool at run time from the running transcript, looping until it calls `finish()` or hits a step budget. Most powerful, most expensive, route unknown in advance.

**The Tool Contract** (what Part 19 lacked): every tool is declared with a JSON schema (name, typed required parameters, description). A validator runs before any tool fires: it rejects an unknown tool and a malformed argument and turns the rejection into an Observation the loop can recover from, instead of a raw crash.

**Continuity**: the corpus is the support-bot world carried from RAG (refund policy, the E-4042 error, and the Acme to Globex acquisition plus earbuds-warranty chain), so the multi-hop question is the same one RAG Part 10 toured and Part 19 ran.

> **Runs with no network, no API key, and no third-party dependency.** Set `RAG_REAL_EMBED=1` (with `sentence-transformers` installed) to opt into the real dense retriever -- it changes only the printed scores. Set `OPENAI_API_KEY` to see the real LLM controller banner; the loop always falls through to the deterministic policy so output stays reproducible.

Companion script: `augmented_llm_loop.py`

## Setup

Two standard-library imports do the work: `os` lets us check for an API key without ever requiring one, and `re` powers the lexical retriever's tokenizer, the calculator's input guard, and the controller's rule matching. No third-party package is needed for the default path -- `numpy` and `sentence-transformers` are imported only inside the optional real retriever, so every cell runs fully offline.

In [ ]:
import os
import re

print("ready -- no network, no API key, no third-party dependency required")

## Step 0 -- Two tiny, eyeball-able corpora

`POLICY_KB` is the support corpus carried from RAG Parts 6-19: refunds, the E-4042 error code, shipping, warranty. `PRODUCTS` is a small new source describing an acquisition (Acme to Globex) and the earbuds warranty chain. It is deliberately split so that no single chunk holds both "who acquired Acme" and "the earbuds warranty" -- that gap is exactly what forces a second retrieval hop, which is what separates rung 1 (one call) from rung 3 (the loop). Short docs, so each doc is its own chunk.

In [ ]:
POLICY_KB = [
    "Refunds are accepted within 30 days of purchase, provided the item is unused and in its original packaging.",
    "To start a return, email support@example.com with your order number. Refunds are processed within five business days of us receiving the item.",
    "Error E-4042 means the payment was declined by the bank; ask the customer to retry with a different card or contact their bank.",
    "Standard shipping takes 3 to 5 business days. Express shipping arrives the next business day.",
    "All electronics include a one-year limited warranty covering manufacturing defects.",
]

PRODUCTS = [
    "Acme Corp was acquired by Globex in 2024.",
    "Globex now manufactures the wireless earbuds product line it inherited from Acme.",
    "Globex-branded wireless earbuds carry a 2-year limited warranty.",
    "The wireless earbuds deliver up to 8 hours of battery life, and up to 24 hours with the charging case.",
]

print(f"Policy KB: {len(POLICY_KB)} chunks. Products: {len(PRODUCTS)} chunks.")

## Step 1 -- Retrieval, with a transparent deterministic default

The default retriever is purely lexical: score each chunk by content-word overlap with the query, normalized by the geometric mean of the two token counts (a cosine-flavored score in [0, 1]). Crude, but deterministic, model-free, network-free, and enough to land every tool call on the right chunk, so the demo's output is reproducible. The real `sentence-transformers` path (from RAG Part 6) is one env flag away and changes only the printed scores.

The tokenizer strips grammatical stop-words so a query's content words (refund, earbuds, warranty) drive the score instead of "what / is / the". A crude plural-s stemmer lets "earbuds" and "earbud" hash to the same content word; a real dense model needs no such hint.

In [ ]:
_TOKEN_RE = re.compile(r"[a-z0-9]+")
_STOPWORDS = {
    "what", "is", "are", "the", "of", "a", "an", "for", "on", "in", "to", "how",
    "do", "does", "and", "my", "i", "there", "with", "your", "our", "who", "by",
    "that", "made", "company", "s", "whats",
}


def _stem(tok):
    """Crudest possible stemmer: drop a trailing plural 's' so 'earbuds' and
    'earbud' hash to the same content word."""
    return tok[:-1] if len(tok) > 3 and tok.endswith("s") else tok


def _tokens(text):
    return [_stem(t) for t in _TOKEN_RE.findall(text.lower()) if t not in _STOPWORDS]


print(_tokens("what is the warranty on the earbuds"))

### The lexical retriever class and the real dense path

`_LexicalRetriever` scores `overlap / sqrt(|q_tokens| * |c_tokens|)`, a cosine-flavored overlap that keeps scores in a readable band without a 90 MB model download. `load_real_retriever()` tries the real `sentence-transformers` path and falls back transparently to the lexical stand-in, printing exactly once. The lexical retriever is the default so this notebook's output is reproducible whether or not a model happens to be cached. Set `RAG_REAL_EMBED=1` to opt into the dense path; it changes only the printed scores, not which chunks are retrieved.

In [ ]:
class _LexicalRetriever:
    """Deterministic, model-free stand-in for a dense retriever (RAG Part 6)."""

    def __init__(self, corpus):
        self.chunks = list(corpus)
        self._chunk_tokens = [set(_tokens(c)) for c in self.chunks]

    def _score(self, q_tokens, c_tokens):
        if not q_tokens or not c_tokens:
            return 0.0
        overlap = len(q_tokens & c_tokens)
        denom = (len(q_tokens) * len(c_tokens)) ** 0.5   # cosine-flavored, in [0,1]
        return overlap / denom

    def retrieve(self, query, k=1):
        q_tokens = set(_tokens(query))
        scored = [
            (self.chunks[i], self._score(q_tokens, self._chunk_tokens[i]))
            for i in range(len(self.chunks))
        ]
        scored.sort(key=lambda x: -x[1])
        return scored[:k]


def load_real_retriever(corpus):
    """Real sentence-transformers retriever; transparent lexical fallback.

    The deterministic lexical retriever is the DEFAULT so output is reproducible.
    Set RAG_REAL_EMBED=1 (with sentence-transformers installed) to opt into the
    real dense path -- only the printed scores change.
    """
    if not os.environ.get("RAG_REAL_EMBED"):
        if not load_real_retriever._announced:
            print("[embed] using deterministic lexical retriever (offline default; "
                  "set RAG_REAL_EMBED=1 for sentence-transformers)")
            load_real_retriever._announced = True
        return _LexicalRetriever(corpus)
    try:
        from sentence_transformers import SentenceTransformer
        import numpy as _np

        model = SentenceTransformer("all-MiniLM-L6-v2")

        class _DenseRetriever:
            def __init__(self):
                self.chunks = list(corpus)
                self.vectors = _np.asarray(
                    model.encode(self.chunks, normalize_embeddings=True))

            def retrieve(self, query, k=1):
                q = _np.asarray(model.encode([query], normalize_embeddings=True))[0]
                scores = self.vectors @ q
                top = _np.argsort(-scores)[:k]
                return [(self.chunks[i], float(scores[i])) for i in top]

        if not load_real_retriever._announced:
            print("[embed] using sentence-transformers (all-MiniLM-L6-v2)")
            load_real_retriever._announced = True
        return _DenseRetriever()
    except Exception as exc:
        if not load_real_retriever._announced:
            print(f"[embed] sentence-transformers unavailable ({type(exc).__name__}); "
                  "using deterministic lexical fallback")
            load_real_retriever._announced = True
        return _LexicalRetriever(corpus)


load_real_retriever._announced = False
_POLICY_STORE = load_real_retriever(POLICY_KB)
_PRODUCTS_STORE = load_real_retriever(PRODUCTS)

## Step 2 -- The Tool Contract: each tool is a typed JSON schema plus a function

This is the layer RAG Part 19 lacked. A tool is declared with the same JSON schema a real LLM is handed for tool calling: a name, a one-line description, and typed required parameters. The four tools cover the three question shapes in this demo: two retrieval tools (`search_policy`, `search_products`) that wrap their own index store, a `calculator` for arithmetic (proving not every question needs retrieval), and `finish` to terminate the loop. `TOOL_SCHEMAS` is the action space, declared once. The same object drives both the offline validator (Step 3) and the real-LLM prompt in `build_prompt()` (Step 4) -- the offline contract and the real one are the same object.

The calculator only lets digits, operators, parentheses, spaces, and a decimal point reach `eval`. A real agent sandboxes tools more thoroughly; this is the minimal guard that keeps `eval` from being a foot-gun while staying one readable line.

In [ ]:
def search_policy(query):
    """Retrieve the single best chunk from the POLICY index."""
    text, score = _POLICY_STORE.retrieve(query, k=1)[0]
    return text, score


def search_products(query):
    """Retrieve the single best chunk from the PRODUCTS index."""
    text, score = _PRODUCTS_STORE.retrieve(query, k=1)[0]
    return text, score


_CALC_RE = re.compile(r"^[\d\s+\-*/().%]+$")


def calculator(expression):
    """Evaluate simple arithmetic. Proves not everything is retrieval."""
    if not _CALC_RE.match(expression):
        return "calculator error: expression contains unsupported characters"
    try:
        return eval(expression, {"__builtins__": {}}, {})   # guarded: digits/ops only
    except Exception as exc:
        return f"calculator error: {type(exc).__name__}"


def finish(answer):
    """Terminate the loop with the final answer."""
    return answer


# TOOL_SCHEMAS is the action space, declared once. Each entry is exactly what a
# real LLM tool-calling API would receive: name -> {description, parameters}.
# parameters maps an arg name -> {type, required}. The same object drives both
# the offline validator AND the real-LLM prompt in build_prompt().
TOOL_SCHEMAS = {
    "search_policy": {
        "description": "search the support/policy index (refunds, errors, shipping, warranty)",
        "parameters": {"query": {"type": "string", "required": True}},
        "fn": search_policy,
    },
    "search_products": {
        "description": "search the products index (acquisitions, product warranties)",
        "parameters": {"query": {"type": "string", "required": True}},
        "fn": search_products,
    },
    "calculator": {
        "description": "evaluate simple arithmetic",
        "parameters": {"expression": {"type": "string", "required": True}},
        "fn": calculator,
    },
    "finish": {
        "description": "return the final answer and stop",
        "parameters": {"answer": {"type": "string", "required": True}},
        "fn": finish,
    },
}

_PY_TYPE = {"string": str, "number": (int, float), "boolean": bool}

print("Tools defined:", ", ".join(TOOL_SCHEMAS))

## Step 3 -- The validator: check a call against its schema before it fires

`validate_call` returns `(ok, error_message)`. It catches the three failures that RAG Part 19 would have crashed on: an unknown tool name, a missing required argument, and an argument of the wrong type. `call_tool` wraps the validator so the caller never has to special-case a crash: a rejected call returns the validator's error message as the observation text, which the loop can read and recover from on the next step. In the agent loop (Step 6), a rejection becomes an Observation rather than a stack trace.

In [ ]:
def validate_call(name, args):
    if name not in TOOL_SCHEMAS:
        return False, f"unknown tool '{name}' (not in the action space)"
    schema = TOOL_SCHEMAS[name]["parameters"]
    for arg_name, spec in schema.items():
        if spec.get("required") and arg_name not in args:
            return False, f"{name} is missing required arg '{arg_name}'"
    for arg_name, value in args.items():
        if arg_name not in schema:
            return False, f"{name} got unexpected arg '{arg_name}'"
        expected = schema[arg_name]["type"]
        if not isinstance(value, _PY_TYPE[expected]):
            got = type(value).__name__
            return False, f"{name} arg '{arg_name}' must be {expected}, got {got}"
    return True, None


def call_tool(name, args):
    """Validate, then invoke. Returns (observation_text, score_or_None).

    A retrieval tool returns (text, score); calculator/finish return (value,
    None). An invalid call returns the validator's error as the observation, so
    the caller never has to special-case a crash.
    """
    ok, err = validate_call(name, args)
    if not ok:
        return f"[invalid call] {err}", None
    result = TOOL_SCHEMAS[name]["fn"](**args)
    if name in ("search_policy", "search_products"):
        text, score = result
        return text, score
    return result, None


print("validate_call and call_tool ready.")

## Step 4 -- generate() and build_prompt(): the real LLM-driven path (reference shape only)

In production the controller is an LLM: you hand it the goal, the running transcript, and `TOOL_SCHEMAS` as a rendered action space, and it emits the next Thought plus Action. `build_prompt()` shows that shape. `generate()` is the single swappable call that lights up the real path. The offline demo never calls it: the deterministic controller in Step 5 is the source of truth for every Thought and Action you see in the output.

The Anthropic alternative is included in comments. SDK names and model IDs move fast; check current docs. Only `generate()` needs edits to go live.

In [ ]:
def _render_schemas():
    lines = []
    for name, s in TOOL_SCHEMAS.items():
        params = ", ".join(f"{p}: {spec['type']}" for p, spec in s["parameters"].items())
        lines.append(f"  {name}({params}) -- {s['description']}")
    return "\n".join(lines)


SYSTEM = """You are an augmented LLM: a model that can call tools. Solve the goal
by reasoning step by step. At each step output exactly:
  Thought: <your reasoning>
  Action: <tool>(<json args>)
Only call tools from this action space (arguments must match the schema):
{schemas}
Call finish(...) as soon as you can answer the goal."""


def build_prompt(goal, transcript):
    """The prompt a REAL LLM controller would see: system + schemas + transcript."""
    history = transcript if transcript else "(no steps yet)"
    return (SYSTEM.format(schemas=_render_schemas())
            + f"\n\nGoal: {goal}\n\nTranscript so far:\n{history}\n\nNext step:")


def generate(prompt):
    """REAL path: ask a hosted LLM for the next step. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


if os.environ.get("OPENAI_API_KEY"):
    print("[controller] OPENAI_API_KEY set; the real LLM controller would drive "
          "generate(build_prompt(...)). Falling through to the deterministic policy "
          "so output stays reproducible.")
else:
    print("[controller] no OPENAI_API_KEY; using deterministic rule-based "
          "controller (offline default)")

## Step 5 -- The deterministic controller: the offline source of truth

Given the goal and the steps taken so far, the controller returns the next step as `(thought, tool_name, args_dict)`. This is a transparent rule policy you can read, keyed on the same cheap signals an LLM would weigh. A real system swaps this body for one `generate()` call against `build_prompt()`; the loop in Step 6 is identical either way.

Three branches cover the three question shapes: arithmetic (calculator then finish), multi-hop acquisition plus warranty (two product retrievals, where hop 2's query is built from hop 1's observation -- that chaining is what separates rung 1 from rung 3), and a policy question (policy index then finish). The controller reads `steps` to know which hop it is on, just as an LLM reads the running transcript.

In [ ]:
_PERCENT_RE = re.compile(r"(\d+(?:\.\d+)?)\s*%\s*of\b[^\d]*\$?\s*(\d+(?:\.\d+)?)")


def _acquirer_from(observation):
    m = re.search(r"acquired by (\w+)", observation)
    return m.group(1) if m else "the acquirer"


def controller(goal, steps):
    """Decide the next step deterministically. `steps` = prior
    (thought, tool, args, observation) tuples. Returns (thought, tool, args)."""
    g = goal.lower()
    n = len(steps)

    # No-retrieval branch: arithmetic -> calculator, then finish.
    pct = _PERCENT_RE.search(g)
    if pct or ("%" in g and "of" in g) or "calculate" in g:
        if n == 0:
            expr = f"{float(pct.group(1)) / 100} * {pct.group(2)}"
            return ("This is arithmetic, not a lookup; use the calculator.",
                    "calculator", {"expression": expr})
        value = steps[-1][3]
        return ("I have the computed value; finish.",
                "finish", {"answer": f"18% of a $250 order is ${float(value):.2f}."})

    # Multi-hop branch: an acquisition + downstream warranty question. No single
    # chunk holds both facts, so hop 2's query is built from hop 1's observation.
    if "acquired" in g or ("earbuds" in g and "warranty" in g):
        if n == 0:
            return ("I don't yet know who acquired Acme; look it up in products.",
                    "search_products", {"query": "who acquired Acme"})
        if n == 1:
            acquirer = _acquirer_from(steps[0][3])
            return (f"Acme was acquired by {acquirer}; now find {acquirer}'s earbuds warranty.",
                    "search_products", {"query": f"{acquirer} earbuds warranty"})
        acquirer = _acquirer_from(steps[0][3])
        return ("I have the warranty term for the earbuds; finish.",
                "finish", {"answer": f"The earbuds are made by {acquirer} (which acquired Acme), "
                                     "and they carry a 2-year limited warranty."})

    # Routing branch: a policy question -> the POLICY index.
    if n == 0:
        sub = re.sub(r"^(what'?s?|what is)\s+(our\s+)?", "", g).strip(" ?")
        return ("This is a policy question; search the policy index.",
                "search_policy", {"query": sub or goal})
    return ("The policy chunk answers the question; finish.",
            "finish", {"answer": steps[-1][3]})


print("controller ready.")

## Step 6 -- The three rungs of the ladder

All three rungs are driven by the same controller. Two helper functions handle display: `_obs_text` formats a retrieval observation with its score, `_json_args` and `_show_args` render argument dicts in a readable JSON-like style. `_retrievals` counts how many steps actually hit an index, for the run footers.

**Rung 1 (`augmented_call`)**: one tool call, then the model must answer. It peeks at what step 2 would be; if the controller would call another search, the question requires chaining that one round cannot do.

**Rung 2 (`fixed_workflow`)**: a hardcoded retrieve-retrieve-synthesize. Correct for this question shape, but the two-hop route is wired at author time -- ask a different question and the sequence no longer fits.

**Rung 3 (`run_agent`)**: the reason/act/observe loop. The controller picks the next action from the transcript at run time; every call is validated before it fires; the loop stops on `finish()` or the step budget. The budget is the honest infinite-loop guard: agentic loops can repeat, oscillate, or never decide they are done, so a hard cap is always present in production.

In [ ]:
def _obs_text(observation, score):
    return f"{observation} (score={score:.2f})" if score is not None else f"{observation}"


def augmented_call(goal):
    """RUNG 1: one augmented call. The model may issue ONE round of tool calls,
    then must answer. It cannot chain a step that depends on the previous one."""
    thought, tool, args = controller(goal, [])
    if tool == "finish":                       # nothing to look up; answer directly
        print(f"  Direct answer (no tool needed):")
        print(f"  ANSWER: {args['answer']}")
        return
    obs, score = call_tool(tool, args)
    arg_str = list(args.values())[0]
    print(f'  Round 1 tool call: {tool}("{arg_str}")')
    print(f"    -> {_obs_text(obs, score)}")
    # Does ONE round settle it? Peek at what step 2 would be: if the controller
    # would call another search, hop 2 depends on hop 1 -> one call cannot do it.
    next_thought, next_tool, next_args = controller(goal, [(thought, tool, args, obs)])
    if next_tool == "finish":
        print(f"  Must answer now (single round):")
        print(f"  ANSWER: {next_args['answer']}")
    else:
        print(f"  Must answer now (no second round):")
        print("  ANSWER: I found that Globex acquired Acme, but answering the warranty needs a")
        print("          SECOND lookup (Globex earbuds warranty) that one round cannot make.")
        print("  -> Incomplete: hop 2 depends on hop 1's result. One call cannot chain.")


def fixed_workflow(goal):
    """RUNG 2: a hardcoded retrieve -> retrieve -> synthesize, wired at author
    time. Correct for THIS question shape, rigid for any other."""
    obs1, s1 = call_tool("search_products", {"query": "who acquired Acme"})
    print(f'  Step 1: search_products("who acquired Acme")    -> {_obs_text(obs1, s1)}')
    acquirer = _acquirer_from(obs1)
    obs2, s2 = call_tool("search_products", {"query": f"{acquirer} earbuds warranty"})
    print(f'  Step 2: search_products("{acquirer} earbuds warranty") -> {_obs_text(obs2, s2)}')
    print(f"  ANSWER: The earbuds are made by {acquirer} (which acquired Acme), and they "
          "carry a 2-year limited warranty.")
    print("  -> Correct -- but the two-hop route was wired at author time. Ask a different")
    print("     shape of question and this exact sequence no longer fits.")


def run_agent(goal, max_steps=6, trace=True):
    """RUNG 3: the reason/act/observe loop. The controller picks the next action
    from the transcript; every call is validated before it fires; the loop stops
    on finish() or the step budget (the honest infinite-loop guard)."""
    def log(msg):
        if trace:
            print(msg)

    steps = []                                       # (thought, tool, args, obs)
    for step in range(1, max_steps + 1):
        thought, tool, args = controller(goal, steps)
        log(f"  Step {step}")
        log(f"    Thought: {thought}")

        if tool == "finish":
            ok, err = validate_call(tool, args)
            if not ok:                               # a malformed finish is still caught
                log(f"    Action: {tool}({args}) -> [invalid call] {err}")
                steps.append((thought, tool, args, f"[invalid call] {err}"))
                continue
            log(f'    Action: finish({{"answer": "{args["answer"]}"}}'+')')
            return args["answer"], step, _retrievals(steps)

        obs, score = call_tool(tool, args)           # validates, then invokes
        log(f"    Action: {tool}({_json_args(args)})")
        log(f"    Observation: {_obs_text(obs, score)}")
        steps.append((thought, tool, args, obs))

    log("  -> step budget exhausted without finish(); stopping.")
    return ("Stopped: did not converge within the step budget.",
            max_steps, _retrievals(steps))


def _json_args(args):
    inner = ", ".join(f'"{k}": "{v}"' for k, v in args.items())
    return "{" + inner + "}"


def _show_args(args):
    """Render args as JSON-ish for display: string values quoted, others bare
    (so a malformed {\"expression\": 42} reads honestly as a number, not a string)."""
    parts = [f'"{k}": ' + (f'"{v}"' if isinstance(v, str) else str(v)) for k, v in args.items()]
    return "{" + ", ".join(parts) + "}"


def _retrievals(steps):
    return sum(1 for _t, tool, _a, _o in steps
               if tool in ("search_policy", "search_products"))


print("augmented_call, fixed_workflow, run_agent ready.")

## Demo -- Tool contract validation, then the multi-hop ladder, then the no-retrieval example

Everything below runs fully offline. The demo has three sections:

1. **Tool contract**: four validation samples showing the three failure modes Part 19 would have crashed on (unknown tool, missing required arg, wrong type) and one valid call.
2. **The ladder on the multi-hop question**: the same goal on all three rungs, showing where rung 1 fails (cannot chain), where rung 2 is correct but rigid (wired at author time), and where rung 3 succeeds by deciding the route at run time.
3. **When not to climb**: the arithmetic example solved on rung 1, showing that a loop is not always necessary.

In [ ]:
line = "=" * 72
print(line)
print("THE AUGMENTED LLM  -  one primitive, three rungs of the agent ladder")
print(line)

if os.environ.get("OPENAI_API_KEY"):
    print("[controller] OPENAI_API_KEY set; the real LLM controller would drive "
          "generate(build_prompt(...)). Falling through to the deterministic policy "
          "so output stays reproducible.")
else:
    print("[controller] no OPENAI_API_KEY; using deterministic rule-based "
          "controller (offline default)")

print("\nTools (the action space, each with a typed schema):")
print("  - search_policy(query: string)      search the support/policy index")
print("  - search_products(query: string)    search the products index")
print("  - calculator(expression: string)    evaluate arithmetic")
print("  - finish(answer: string)            return the final answer and stop")

# --- The tool contract: validation before any call fires. -------------------
print("\n" + "-" * 72)
print("THE TOOL CONTRACT: validate every call BEFORE it fires.")
print("-" * 72)
samples = [
    ("search_products", {"query": "who acquired Acme"}),
    ("search_web", {"query": "..."}),
    ("calculator", {"expr": "0.18 * 250"}),
    ("calculator", {"expression": 42}),
]
for name, args in samples:
    ok, err = validate_call(name, args)
    verdict = "OK" if ok else f"REJECTED: {err}"
    print(f"  {name}({_show_args(args)})".ljust(53) + f"-> {verdict}")
print("  Part 19 had no such layer: each of these reached the function and crashed "
      "(or worse, ran).")

multihop = "what is the warranty on the earbuds made by the company that acquired Acme?"
print("\n" + line)
print("THE LADDER: same multi-hop question, three rungs.")
print(f"GOAL: {multihop}")
print(line)

print("\n" + "-" * 72)
print("RUNG 1 - ONE AUGMENTED CALL: tools available, but a single round then answer.")
print("-" * 72)
augmented_call(multihop)

print("\n" + "-" * 72)
print("RUNG 2 - FIXED WORKFLOW: a hardcoded retrieve -> retrieve -> synthesize.")
print("-" * 72)
fixed_workflow(multihop)

print("\n" + "-" * 72)
print("RUNG 3 - FULL AGENT: reason/act/observe; the route is decided at run time.")
print("-" * 72)
answer, steps_taken, hits = run_agent(multihop)
print(f"  ANSWER: {answer}")
print(f"  ({steps_taken} steps, {hits} retrievals -- same correct answer as the workflow, but NO route was")
print("   wired in advance; the agent chose each step from the transcript.)")

arithmetic = "what is 18% of a $250 order?"
print("\n" + "-" * 72)
print("WHEN NOT TO CLIMB: a no-retrieval question needs no agent at all.")
print(f"GOAL: {arithmetic}")
print("-" * 72)
print("  RUNG 1 - one augmented call:")
augmented_call(arithmetic)
print("  -> Solved on the cheapest rung. Don't reach for the loop when one call answers.")

print("\n" + line)
print("Done. The ladder, bottom to top:")
print("  - one augmented call : cheapest; cannot chain dependent steps (the multi-hop gap)")
print("  - fixed workflow     : cheap + predictable; route wired at author time")
print("  - full agent         : route decided at RUN time; needed only when the path is")
print("                         not known in advance -- and every tool call is validated")
print("                         against its schema before it fires.")
print("Reach for the lowest rung that works.")
print(line)